# QuestDB 10 Demo

This demo will demonstrate how to use the latest `questdb` client (5.0.0) and taking advantage of the new bi-directional [QWP protocol](https://questdb.com/blog/questdb-10-release) to query the database.  
QuestDB is still compatible with the [PGWire protocol](https://questdb.com/docs/connect/compatibility/pgwire/python) and the [InfluxDB Line Protocol](https://questdb.com/docs/connect/compatibility/ilp/overview).

A web interface is available on [port 9000](http://localhost:9000/index.html) when the instance is running. Data and dashboards can be managed from this page.  
More information about the [web console page](https://questdb.com/docs/getting-started/web-console/overview).

## 1. Prerequistes
### 1.1 QuestDB >= 10
The official image is available on [hub.docker.com](https://hub.docker.com/r/questdb/questdb).  
The documentation for all platforms is available on [Using Docker with QuestDB](https://questdb.com/docs/deployment/docker).

### 1.2 Python >= 3.10
The client requires python >= 3.10. In order to run this demo, other packages need to be installed:   
`python -m pip install -U questdb pandas pyarrow matplot`  

More information about the [Python client for QuestDB](https://questdb.com/docs/connect/clients/python).

## 2. Configuration
More information about the [connection string](https://questdb.com/docs/connect/clients/connect-string/#key-index).

In [ ]:
from datetime import datetime, UTC
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import questdb
from questdb import TimestampMicros

CONF_STR = "ws::addr=localhost:9000;"
REPO = "quest_test"
version = datetime.now(UTC).replace(hour=0, minute=0, second=0, microsecond=0)

## 3. Creating tables

Tables in QuestDB can be created explicitly or on the fly during data ingestion.

In [ ]:
with questdb.connect(CONF_STR) as db:
    db.execute(
        f"DROP TABLE IF EXISTS {REPO};"
    )
    db.execute(
        f"CREATE TABLE IF NOT EXISTS {REPO} (index_name SYMBOL INDEX, version TIMESTAMP, time TIMESTAMP, value DOUBLE, range DOUBLE[][]) TIMESTAMP(time) PARTITION BY MONTH TTL 100 YEARS DEDUP UPSERT KEYS (index_name, version, time);"
    )

`SYMBOL`: Indexable string  
`PARTITION`: YEAR | MONTH | WEEK | DAY | HOUR  
`TTL`: Data retention  
`DEDUP UPSERT KEY`: Upsert values using the upsert keys

More information about the supported [Data Types](https://questdb.com/docs/query/datatypes/overview).

## 4. Ingestion
### 4.1 Streaming ingestion
Only float64 are supported for multi-dimentional arrays.

In [ ]:
index_name = f"streaming_{datetime.now(UTC).strftime('%Y%m%d_%H%M%S')}"

with questdb.connect(CONF_STR) as db, db.sender() as sender:
    sender.row(
        REPO,
        symbols={"index_name": index_name},
        columns={
            "value": 2.0,
            "range": np.array([[1.0, 3.0]], dtype=np.float64),
            "version": version,
        },
        at=TimestampMicros.now(),
    )

    sender.row(
        REPO,
        symbols={"index_name": index_name},
        columns={
            "value": 666.666,
            "range": np.array([[555.555, 777.777]], dtype=np.float64),
            "version": version,
        },
        at=TimestampMicros.now(),
    )

    sender.flush(wait=True)

### 4.2 Batch ingestion
`df` accepts pandas/polars DataFrame and everything implementing the Arrow Data Interface ([DF ingestion](https://questdb.com/docs/connect/clients/python/#dataframe-ingestion)).  
In the following code `pandas` and `pyarrow` are used to upload a huge dataset.  
-> In our case, the size of the dataset is limited by the size of the Arrow array length (max 16777216)


In [ ]:
index_name = f"bulk_{datetime.now(UTC).strftime('%Y%m%d_%H%M%S')}"

def generate_periodic_data(years_to_generate, intervals_per_year) -> np.ndarray:
    total_intervals = intervals_per_year * years_to_generate
    intervals = np.arange(total_intervals)
    trend = 0.0005 * intervals
    seasonal = 10 * np.sin(2 * np.pi * (intervals - (80 * 24 * 4)) / intervals_per_year)
    np.random.seed(42)
    noise = np.random.normal(loc=0.0, scale=1.5, size=total_intervals)
    return trend + seasonal + noise

data_150y_15m = generate_periodic_data(150, 365 * 24 * 4) #param for 15 min resolution
total_points = len(data_150y_15m)

df = pd.DataFrame({
    "index_name": pd.Categorical([index_name] * total_points),
    "value": data_150y_15m,
    "range": [np.array([[0.0000001, 10000000.0]], dtype=np.float64)] * total_points,
    "version": [version] * total_points,
    "timestamp": pd.date_range(start=datetime(1970, 1, 1, tzinfo=UTC), periods=total_points, freq="15min")
})

with questdb.connect(CONF_STR) as db:
    db.dataframe(df, table_name=REPO, symbols=["index_name"], at="timestamp")

## 5. Querying
### 5.1 Filtering

In [ ]:
df = None
sql = f"SELECT time, value FROM {REPO} WHERE index_name = $1 AND time >= $2 AND time <= $3;"
params = [index_name, datetime(2000, 1, 1, tzinfo=UTC), datetime(2024, 12, 31, 23, 59, 59, tzinfo=UTC)]
with questdb.connect(CONF_STR) as db, db.query(sql, params) as result:
    df = result.to_pandas()

df.plot(x="time", y="value", title=f"Filtering {len(df)} points", xlabel="Time", ylabel="Value")
plt.show()

### 5.2 Filtering, resampling and aggregating
More information about the [Aggregate functions](https://questdb.com/docs/query/functions/aggregation).    
More information about the [Resampling functions](https://questdb.com/docs/query/sql/sample-by).

In [ ]:
df = None
sql = f"SELECT time, avg(value) as avg_value, min(value) as min_value, max(value) as max_value FROM {REPO} WHERE index_name = $1 AND time >= $2 AND time <= $3 SAMPLE BY 1d;"
params = [index_name, datetime(2000, 1, 1, tzinfo=UTC), datetime(2024, 12, 31, 23, 59, 59, tzinfo=UTC)]
with questdb.connect(CONF_STR) as db, db.query(sql, params) as result:
    df = result.to_pandas()

df.plot(x="time", y=["avg_value", "min_value", "max_value"], title=f"Resampling and aggregating {len(df)} points", xlabel="Time", ylabel="Value")
plt.show()